# Exploratory Data Analysis (EDA) on Superstore Retail Sales

**Task 1 | Tools:** Python, Pandas, Matplotlib, Seaborn

## Objective
Analyze customer purchase volume, monthly sales seasonality, category-wise revenue, regional and customer-segment performance, and discount sensitivity. The notebook concludes with at least five actionable business findings.

> **Dataset note:** Keep `superstore_sales.csv` in the same folder as this notebook. The supplied CSV contains 10,800 rows, including 9,994 transaction rows and additional appended records that do not contain transaction dates. The cleaning step below isolates valid transaction records rather than treating those appended records as sales.


## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_FILE = Path("superstore_sales.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Place superstore_sales.csv in the same folder as this notebook."
    )

raw = pd.read_csv(DATA_FILE)
print("Raw dataset shape:", raw.shape)
display(raw.head())
print("\\nColumn names:")
print(raw.columns.tolist())

## 2. Data quality assessment and cleaning

In [ ]:
# Review missing values before cleaning
missing_summary = raw.isna().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0].to_frame("Missing values"))

# The supplied file has transaction rows with Order Date populated.
# Rows without an Order Date are appended non-transaction records and are excluded.
df = raw.loc[raw["Order Date"].notna()].copy()

# Parse dates and convert numeric fields safely
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")
numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove records missing essential analysis fields
required = ["Order Date", "Order ID", "Customer ID", "Category",
            "Region", "Segment", "Sales", "Quantity", "Discount", "Profit"]
before = len(df)
df = df.dropna(subset=required).copy()
removed = before - len(df)

# Remove exact duplicate rows, if any
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

# Extract calendar dimensions
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.strftime("%b")
df["Year-Month"] = df["Order Date"].dt.to_period("M").astype(str)

print(f"Raw rows: {len(raw):,}")
print(f"Rows without Order Date excluded: {raw['Order Date'].isna().sum():,}")
print(f"Rows removed for missing required fields: {removed:,}")
print(f"Exact duplicate rows removed: {duplicate_count:,}")
print("Cleaned dataset shape:", df.shape)
print("Date range:", df["Order Date"].min().date(), "to", df["Order Date"].max().date())
display(df.head())

## 3. Exploratory summary and customer purchase volume

In [ ]:
summary = df[["Sales", "Quantity", "Discount", "Profit"]].describe().T
display(summary)

order_summary = df.groupby("Order ID").agg(
    Order_Sales=("Sales", "sum"),
    Order_Quantity=("Quantity", "sum"),
    Order_Profit=("Profit", "sum"),
    Customer_ID=("Customer ID", "first")
)
customer_summary = df.groupby("Customer ID").agg(
    Orders=("Order ID", "nunique"),
    Purchase_Quantity=("Quantity", "sum"),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).sort_values("Total_Sales", ascending=False)

print("Unique orders:", df["Order ID"].nunique())
print("Unique customers:", df["Customer ID"].nunique())
print("Average sales per order: $", round(order_summary["Order_Sales"].mean(), 2))
print("Average quantity per order:", round(order_summary["Order_Quantity"].mean(), 2))
display(customer_summary.head(10))

## 4. Monthly sales seasonality

In [ ]:
monthly_sales = df.groupby("Year-Month", as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)
monthly_sales["Year-Month-Date"] = pd.to_datetime(monthly_sales["Year-Month"])

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_sales, x="Year-Month-Date", y="Sales", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

monthly_seasonality = df.groupby("Month").agg(
    Sales=("Sales", "sum"),
    Average_Monthly_Sales=("Sales", "mean"),
    Orders=("Order ID", "nunique")
).reindex(range(1, 13))
monthly_seasonality.index = pd.Index(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], name="Month"
)
display(monthly_seasonality)

plt.figure(figsize=(10, 4))
sns.barplot(x=monthly_seasonality.index, y=monthly_seasonality["Sales"])
plt.title("Total Sales by Calendar Month (All Years)")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

## 5. Category-wise and sub-category performance

In [ ]:
category_perf = df.groupby("Category").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"),
    Orders=("Order ID", "nunique")
).sort_values("Sales", ascending=False)
category_perf["Profit Margin (%)"] = category_perf["Profit"] / category_perf["Sales"] * 100
display(category_perf)

plt.figure(figsize=(8, 4))
sns.barplot(data=category_perf.reset_index(), x="Category", y="Sales")
plt.title("Revenue by Product Category")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

subcategory_perf = df.groupby(["Category", "Sub-Category"]).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum")
).sort_values("Profit")
display(subcategory_perf)

plt.figure(figsize=(12, 7))
sns.barplot(data=subcategory_perf.reset_index(), y="Sub-Category", x="Profit")
plt.title("Profit by Sub-Category")
plt.tight_layout()
plt.show()

## 6. Region and customer-segment analysis

In [ ]:
region_perf = df.groupby("Region").agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"), Orders=("Order ID", "nunique")
).sort_values("Sales", ascending=False)
region_perf["Profit Margin (%)"] = region_perf["Profit"] / region_perf["Sales"] * 100
display(region_perf)

segment_perf = df.groupby("Segment").agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"), Orders=("Order ID", "nunique"),
    Customers=("Customer ID", "nunique")
).sort_values("Sales", ascending=False)
segment_perf["Profit Margin (%)"] = segment_perf["Profit"] / segment_perf["Sales"] * 100
display(segment_perf)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=region_perf.reset_index(), x="Region", y="Sales", ax=axes[0])
axes[0].set_title("Sales by Region")
sns.barplot(data=segment_perf.reset_index(), x="Segment", y="Sales", ax=axes[1])
axes[1].set_title("Sales by Customer Segment")
plt.tight_layout()
plt.show()

## 7. Discount sensitivity: discount versus profit

In [ ]:
discount_perf = df.groupby("Discount").agg(
    Transactions=("Discount", "size"),
    Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Average_Profit=("Profit", "mean")
).sort_index()
display(discount_perf)

plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="Discount", y="Profit", alpha=0.35)
sns.regplot(data=df, x="Discount", y="Profit", scatter=False, color="red")
plt.title("Discount vs. Profit per Transaction")
plt.xlabel("Discount rate")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

discount_profit = df.groupby("Discount")["Profit"].mean()
plt.figure(figsize=(9, 4))
sns.barplot(x=discount_profit.index.astype(str), y=discount_profit.values)
plt.title("Average Profit at Each Discount Level")
plt.xlabel("Discount")
plt.ylabel("Average Profit")
plt.tight_layout()
plt.show()

print("Correlation between discount and profit:",
      round(df["Discount"].corr(df["Profit"]), 3))

## 8. Five annotated business findings

In [ ]:
# Generate findings from the cleaned data so the conclusions remain reproducible.
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
top_category = category_perf["Sales"].idxmax()
top_category_share = category_perf.loc[top_category, "Sales"] / total_sales * 100
top_region = region_perf["Sales"].idxmax()
top_segment = segment_perf["Sales"].idxmax()
best_month = monthly_seasonality["Sales"].idxmax()
worst_profit_discount = discount_perf["Average_Profit"].idxmin()
zero_discount_profit = discount_perf.loc[0.0, "Average_Profit"] if 0.0 in discount_perf.index else np.nan
high_discount = discount_perf.loc[discount_perf.index >= 0.30, "Average_Profit"].mean()

findings = [
    f"**Revenue scale:** The cleaned transactions generated ${total_sales:,.2f} in sales and ${total_profit:,.2f} in profit.",
    f"**Category strength:** {top_category} is the highest-revenue category, contributing {top_category_share:.1f}% of total sales.",
    f"**Regional performance:** {top_region} is the leading region by total sales; compare its profit margin with other regions before allocating more promotional budget.",
    f"**Customer segment:** {top_segment} contributes the highest sales among customer segments. Retention and repeat-purchase campaigns can prioritize this segment while still monitoring profitability.",
    f"**Seasonality:** {best_month} has the highest combined sales across calendar years. Plan inventory and marketing capacity ahead of this seasonal peak.",
    f"**Discount sensitivity:** The discount level with the lowest average transaction profit is {worst_profit_discount:.0%}. Average profit for discounts of 30% or more is ${high_discount:,.2f}, so deep discounts should be reviewed carefully."
]
for i, finding in enumerate(findings, 1):
    print(f"{i}. {finding}\\n")

print("Interpretation caution: these are descriptive associations, not proof that discounts alone cause lower profit. Product mix, category, and transaction size can also influence results.")

## 9. Conclusion and recommendations

1. Use monthly sales patterns to plan inventory, staffing, and campaign timing.
2. Monitor category and sub-category profit alongside revenue; high sales do not always mean high profitability.
3. Compare regional profit margins before shifting marketing or distribution budgets.
4. Use customer-segment sales and repeat-order metrics to guide retention campaigns.
5. Review high-discount transactions and set discount guardrails where average profit is weak.

**Deliverable checklist:** data cleaning, missing-value treatment, date parsing, Year-Month extraction, customer purchase-volume profiling, monthly sales trend, category/region/segment aggregation, discount-profit analysis, visualizations, and more than five business findings are included.
